# From LSTMs to Transformers
## See attention work - a practical 60-minute session

**You already know:** RNNs and LSTMs. **You need:** basic Python. **You do not need:** a GPU or any training.

By the end, you can explain what an encoder and decoder do, read an attention heatmap, and show how a decoder predicts its next token.

| Minutes | What we do |
|---|---|
| 0-5 | Bridge from LSTMs |
| 5-15 | Encoder, decoder, and the context bottleneck |
| 15-35 | Build and change a tiny attention experiment |
| 35-45 | Self-attention, position, and hiding the future |
| 45-55 | Inspect a real model's next-token predictions |
| 55-60 | Three-question recap and discussion |

**Before class:** follow the Session 3 setup in the README, including the model download. Then restart the kernel and run all cells. During class, edit only the marked experiments. Model download time is not part of the hour.

**Presenting this session?** Read the separate [presenter study guide](Session3_Presenter_Study_Guide.md) first. It explains every diagram, the code, expected results, and common questions without adding reading to this one-hour session.

## 1. The bridge from LSTMs [0-5 min]

An LSTM reads a sequence step by step, carrying a hidden state and a cell state. Its gates help retain useful information, but long-distance relationships can still be difficult and the recurrence limits parallel processing across positions.

![LSTM recurrent information path compared with direct attention lookups](assets/transformers/01_recurrence.png)

> **Ask the room:** In "The animal didn't cross the street because it was tired", which words help you interpret "it"?

**Our new idea:** let a token directly combine information from relevant positions instead of relying only on information passed through recurrent steps.

*Presenter cue: take two answers, then move on. We will illustrate information lookup, not claim to solve pronoun resolution.*

## 2. Encoder and decoder are roles [5-15 min]

Consider translating **"I love learning"** into **"J'aime apprendre"**.

![Encoder-decoder translation with a fixed context, compared with attention over encoder states](assets/transformers/02_encoder_decoder.png)

In a basic LSTM encoder-decoder, the input is summarized into a fixed-size context passed to the decoder. That can become a bottleneck for long inputs.

**Attention:** keep the encoder's per-position representations available. At each output step, the decoder can form a different weighted combination of them.

Encoder-decoder models existed **before** Transformers. Attention can also be added to RNN/LSTM models; it is not exclusive to Transformers.

> **Quick check:** Must a five-word input produce a five-word output? **No.** Input and output lengths can differ.

*Presenter cue: explain the roles with the translation example; do not introduce architecture internals yet.*

## 3. Attention is a weighted information lookup [15-35 min]

- **Query (Q):** what am I looking for?
- **Key (K):** what can each position be matched on?
- **Value (V):** what information does that position contribute?

We score query-key matches, turn the scores into weights that sum to 1, then take a weighted sum of the values.

![Query-key matching, scaled scores, softmax weights, and value mixing for the actual notebook example](assets/transformers/03_attention.png)

**These vectors are hand-crafted teaching examples, not learned language understanding.** In a real Transformer, learned projections produce Q, K, and V from token representations.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import display

np.set_printoptions(precision=3, suppress=True)

def softmax(scores):
    shifted = scores - np.max(scores, axis=-1, keepdims=True)
    exp_scores = np.exp(shifted)
    return exp_scores / exp_scores.sum(axis=-1, keepdims=True)

def attention(queries, keys, values, causal=False):
    scores = queries @ keys.T / np.sqrt(keys.shape[-1])
    if causal:
        # Aligned self-attention: row i cannot read columns after i.
        if scores.shape[0] != scores.shape[1]:
            raise ValueError('This demo mask requires aligned, equal-length Q and K.')
        future = np.triu(np.ones(scores.shape, dtype=bool), k=1)
        scores = np.where(future, -np.inf, scores)
    weights = softmax(scores)
    return weights @ values, weights

def show_weights(weights, query_labels, key_labels, title):
    fig, ax = plt.subplots(figsize=(7, max(2, len(query_labels) * 0.65)))
    image = ax.imshow(weights, cmap='Blues', vmin=0, vmax=1, aspect='auto')
    ax.set_xticks(range(len(key_labels)), labels=key_labels)
    ax.set_yticks(range(len(query_labels)), labels=query_labels)
    ax.set(xlabel='Keys: positions read from', ylabel='Queries: positions asking', title=title)
    for row in range(weights.shape[0]):
        for col in range(weights.shape[1]):
            weight = weights[row, col]
            ax.text(col, row, f'{weight:.2f}', ha='center', va='center',
                    color='white' if weight > 0.55 else 'black')
    fig.colorbar(image, ax=ax, label='Attention weight')
    fig.tight_layout()
    plt.show()

print('Ready: only NumPy and Matplotlib are needed for the attention experiments.')


### Experiment A: ask a different question

Imagine three records about an animal, a street, and tiredness. Our two key dimensions loosely mean **entity** and **condition**. The values deliberately use a different space: three record indicators. This separates **matching** from **information returned**.

**Predict first:** will query `[3, 0]` favor "animal" or "tired"? Run the cell. Then change only `query` to `[0, 3]` and run it again.

In [ ]:
labels = ['animal', 'street', 'tired']
keys = np.array([[2.0, 0.0], [0.4, 0.2], [0.0, 2.0]])
values = np.eye(3)
query = np.array([[3.0, 0.0]])  # TRY: [[0.0, 3.0]], then [[0.0, 0.0]]

context, weights = attention(query, keys, values)
print('Scaled matching scores:', query @ keys.T / np.sqrt(keys.shape[-1]))
print('Weights:', weights)
print('Weighted values (context):', context)
print('Sum of weights:', weights.sum())
show_weights(weights, ['our query'], labels, 'Change the query; change the information retrieved')
assert np.allclose(weights.sum(axis=-1), 1.0)


**What should you notice?** `[3, 0]` favors animal; `[0, 3]` favors tired; `[0, 0]` gives equal weights. The output is usually a blend, not a hard selection.

Here the context equals the weights because we chose identity vectors as values. Real value vectors are learned features, not record labels.

**Follow one number:** `[3, 0]` dot `[2, 0]` gives `6`. Divide by `sqrt(2)` to get `4.243`. Softmax across **all three scores** gives animal a weight of about `0.954`. Multiply each value vector by its weight and add the results.

### One formula, now that you have seen it

$$\mathrm{Attention}(Q,K,V)=\mathrm{softmax}\left(\frac{QK^T}{\sqrt{d_k}}\right)V$$

Read it as: **match -> scale -> normalize -> mix**. Scaling helps keep dot-product magnitudes manageable as the key dimension grows. No derivation needed.

*Presenter cue: spend the time on participants predicting and changing the query, not on matrix notation.*

### Experiment B: self-attention

In **self-attention**, queries, keys, and values come from the same sequence. Every position asks its own question.

For readability, use three hand-crafted token vectors and identity projections, so Q = K = V = X. A trained model normally learns separate projections.

**Predict:** which other token should "cat" attend to more: "kitten" or "road"?

In [ ]:
tokens = ['cat', 'kitten', 'road']
X = np.array([[2.0, 0.0], [1.8, 0.2], [0.0, 2.0]])
context, full_weights = attention(X, X, X)
show_weights(full_weights, tokens, tokens, 'Self-attention: each row mixes information from the sequence')
print('Original cat vector:', X[0])
print('Contextualized cat vector:', context[0])
assert full_weights[0, 1] > full_weights[0, 2]
assert np.allclose(full_weights.sum(axis=-1), 1.0)


The first output now mixes information from other positions. It is not just the original token vector.

**Important:** these attractive patterns were designed by us. A real model learns its representations and projections from data. An attention heatmap shows information mixing; it is not a guaranteed explanation of a model's reasoning.

## 4. From attention to a Transformer [35-45 min]

A Transformer combines:

![A simplified decoder-only Transformer and one GPT-2-style block](assets/transformers/05_decoder_block.png)

| Piece | Simple purpose |
|---|---|
| Token embeddings + positional information | Represent token content and order |
| Multi-head attention | Learn several information-mixing patterns |
| Feed-forward network | Transform each position's features |
| Residual connections + normalization | Help information flow and training stability |
| Repeated blocks | Build richer representations |

Position matters: **"dog bites man"** is not **"man bites dog"**. Plain self-attention without positional information or position-dependent masking cannot distinguish order in the usual way.

**No recurrent state passed from token to token inside a standard Transformer block.** During training, many positions can be processed in parallel. Autoregressive generation still produces new tokens step by step.

### Experiment C: hide the future

A next-token model must not peek at future tokens. A **causal mask** makes future-position weights zero.

![Encoder access to both directions versus a decoder's triangular causal mask](assets/transformers/04_causal_mask.png)

The diagonal is allowed: the representation at `love` reads `I love` to predict the **next** token. Reading `love` is not leaking the next token.

**Predict:** which columns can the first row read?

In [ ]:
causal_context, causal_weights = attention(X, X, X, causal=True)
show_weights(causal_weights, tokens, tokens, 'Causal self-attention: no reading to the right')
print('First position can only read itself:', causal_weights[0])
assert np.allclose(np.triu(causal_weights, k=1), 0.0)
assert np.allclose(causal_weights.sum(axis=-1), 1.0)

changed_X = X.copy()
changed_X[-1] = [50.0, -50.0]
changed_context, _ = attention(changed_X, changed_X, changed_X, causal=True)
assert np.allclose(causal_context[:-1], changed_context[:-1])
print('Changing the future did not change earlier outputs.')


### Three common arrangements

| Arrangement | What can it read? | Typical use | Example |
|---|---|---|---|
| Encoder-only | Input context on both sides | Classification, contextual representations | BERT |
| Decoder-only | Current and earlier positions | Next-token generation | GPT-2 |
| Encoder-decoder | Encoder reads input; decoder reads earlier output and attends to encoder output | Translation, summarization | T5 |

**Self-attention:** look within a sequence. **Cross-attention:** decoder queries look up encoder representations.

*Presenter cue: this is the architecture overview, not three extra demos. Move on at minute 45.*

## 5. A real decoder: what comes next? [45-55 min]

We use **DistilGPT2**, a small pretrained English text-completion model. It runs on CPU. It is not a chat assistant, and its completions can be wrong or awkward.

The model assigns a score to each possible next **token**; a token can be a word, part of a word, punctuation, or whitespace. Softmax turns scores into a probability distribution.

![Next-token generation loop: context, vocabulary scores, probabilities, choose and append](assets/transformers/06_generation.png)

**Setup must already be complete.** Loading below uses cached files only, so a missing model raises an explicit error instead of starting a surprise download during class.

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

MODEL_ID = 'distilbert/distilgpt2'
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, local_files_only=True)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID, local_files_only=True, use_safetensors=True
).to('cpu')
model.eval()
print(f'Loaded {MODEL_ID} on CPU. Ready for next-token prediction.')


### Experiment D: change the prompt

**Predict first:** what might follow **"The capital of France is"**?

Run the next cell, then change `prompt` to **"For breakfast, I like to eat"**. These are observations, not a quiz with guaranteed model answers. `repr` below makes spaces inside tokens visible.

In [ ]:
prompt = 'The capital of France is'  # TRY: 'For breakfast, I like to eat'
inputs = tokenizer(prompt, return_tensors='pt')
with torch.inference_mode():
    logits = model(**inputs).logits[0, -1]
    probabilities = torch.softmax(logits, dim=-1, dtype=torch.float64)
    top_probabilities, top_ids = torch.topk(probabilities, k=5)

print('Input token pieces:', [tokenizer.decode([i]) for i in inputs['input_ids'][0].tolist()])
print('\nTop five next-token candidates:')
for token_id, probability in zip(top_ids.tolist(), top_probabilities.tolist()):
    print(f'{tokenizer.decode([token_id])!r:20} {probability:7.2%}')
print(f'Total probability of these five: {top_probabilities.sum().item():.2%}')
print('The remaining probability belongs to the rest of the vocabulary.')
assert torch.isclose(probabilities.sum(), probabilities.new_tensor(1.0))


### Generate one token at a time

At each step: **read the current context -> choose a next token -> append it -> repeat**.

For a predictable demonstration, choose the highest-probability token each time (**greedy decoding**). This is not guaranteed to produce the best sentence. We recompute the short context for clarity; production generation usually caches earlier keys and values.

In [ ]:
generated_ids = inputs['input_ids'].clone()
generated_mask = inputs['attention_mask'].clone()
with torch.inference_mode():
    for step in range(12):
        next_logits = model(input_ids=generated_ids, attention_mask=generated_mask).logits[:, -1, :]
        next_id = torch.argmax(next_logits, dim=-1, keepdim=True)
        generated_ids = torch.cat([generated_ids, next_id], dim=1)
        generated_mask = torch.cat([generated_mask, torch.ones_like(next_id)], dim=1)
        print(f'{step + 1:2}: +{tokenizer.decode(next_id[0].tolist())!r} -> '
              f'{tokenizer.decode(generated_ids[0].tolist(), skip_special_tokens=True)}')
        if next_id.item() == tokenizer.eos_token_id:
            break


**Discuss:** changing a prompt changes the distribution, not just a single fixed answer. Fluent output is not evidence that the text is true.

**Do not promise "Paris" as the result.** This small model may choose a generic continuation or repeat itself. That is still a successful demonstration of next-token prediction, not a factual-answering benchmark.

*Presenter cue: demonstrate one prompt, let participants change it once, then run the token-by-token loop. Do not add sampling, training, fine-tuning, or extra model downloads to this hour.*

## 6. Explain it back [55-60 min]

1. **Why attention?** A position can directly mix information from relevant positions rather than relying only on a recurrent chain.
2. **Why a causal mask?** A next-token model must not read future tokens when learning to predict them.
3. **Encoder versus decoder?** An encoder builds input representations; an autoregressive decoder generates using earlier output. Encoder-decoder Transformers connect the two with cross-attention.

**Final takeaway:** attention mixes information; a Transformer organizes that mechanism into blocks; a language-model head turns the resulting representation into next-token scores.

### References for later (not extra session material)

- [Attention Is All You Need](https://arxiv.org/abs/1706.03762)
- [The Illustrated Transformer](https://jalammar.github.io/illustrated-transformer/)
- [DistilGPT2 model card and limitations](https://huggingface.co/distilbert/distilgpt2)
